# 🗄️ Relational Model & SQLite Build

**Author:** Gabriella Marín  
**Project:** Multi-Layer Water Quality Risk & Regulatory Analytics System
**Phase:** Phase 2 – Relational Modeling & SQL

**Objective:**  
Build a relational SQLite database from Phase 1 outputs to ensure traceability, enable efficient queries,
and prepare the foundation for normative compliance evaluation.

## 1. Load Phase 1 outputs + paths

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

# Paths
PROJECT_DIR = Path(r"D:\Documents\Portfolio\01-water-quality-normative")
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

LONG_CSV = OUTPUT_DIR / "phase1_clean_long.csv"
WIDE_CSV = OUTPUT_DIR / "phase1_sample_wide.csv"
DB_PATH  = DATA_DIR / "water_quality_analysis.db"

if not LONG_CSV.exists():
    raise FileNotFoundError(f"Missing Phase 1 output: {LONG_CSV}")
if not WIDE_CSV.exists():
    raise FileNotFoundError(f"Missing Phase 1 output: {WIDE_CSV}")

# Load outputs
df_long = pd.read_csv(LONG_CSV)
df_wide = pd.read_csv(WIDE_CSV)

print("Loaded long:", df_long.shape)
print("Loaded wide:", df_wide.shape)

display(df_long.head(3))
display(df_wide.head(3))

FileNotFoundError: Missing Phase 1 output: D:\Documents\Portfolio\01-water-quality-normative\data\outputs\phase1_clean_long.csv

## 2. Schema Definition (Relational Model)

The database follows a minimal star-like structure:

- `dim_parameter`: standardized parameters and canonical units  
- `dim_location`: unique monitoring locations  
- `fact_sample`: unique samples (sample id + date + location)  
- `fact_measurement`: long-format measurements linked to samples and parameters

This structure preserves full traceability and supports downstream compliance and risk analysis.

In [ ]:
# Basic column expectations (from Phase 1)
required_long_cols = {
    "CODIGO__MUESTRA", "FECHA", "parameter_std",
    "value_raw", "is_censored", "value_num", "unit_raw"}

missing = required_long_cols - set(df_long.columns)
if missing:
    raise ValueError(f"df_long is missing required columns: {missing}")

# Ensure types
df_long["CODIGO__MUESTRA"] = pd.to_numeric(df_long["CODIGO__MUESTRA"], errors="coerce").astype("Int64")
df_long["FECHA"] = pd.to_datetime(df_long["FECHA"], errors="coerce")
df_long["is_censored"] = df_long["is_censored"].astype(bool)
df_long["value_num"] = pd.to_numeric(df_long["value_num"], errors="coerce")

# dim_parameter
# canonical unit = most frequent unit_raw per parameter
param_unit = (
    df_long.groupby("parameter_std")["unit_raw"]
    .agg(lambda s: s.value_counts(dropna=True).index[0] if len(s.dropna()) else None)
    .reset_index()
    .rename(columns={"unit_raw": "canonical_unit"}))

dim_parameter = param_unit.copy()
dim_parameter.insert(0, "parameter_id", range(1, len(dim_parameter) + 1))

# dim_location (from wide, because it has unique meta per sample)
location_cols = ["NOMBRE DEL PUNTO DE MONITOREO", "LATITUD", "LONGITUD", "DEPARTAMENTO", "MUNICIPIO"]
for c in location_cols:
    if c not in df_wide.columns:
        raise ValueError(f"df_wide missing location column: {c}")

dim_location = (
    df_wide[location_cols]
    .drop_duplicates()
    .reset_index(drop=True))

dim_location.insert(0, "location_id", range(1, len(dim_location) + 1))

# fact_sample
sample_cols = ["CODIGO__MUESTRA", "FECHA"] + location_cols
for c in ["CODIGO__MUESTRA", "FECHA"]:
    if c not in df_wide.columns:
        raise ValueError(f"df_wide missing sample column: {c}")

sample_base = df_wide[sample_cols].copy()
sample_base["FECHA"] = pd.to_datetime(sample_base["FECHA"], errors="coerce")

fact_sample = sample_base.merge(dim_location, on=location_cols, how="left")

# fact_measurement
# link long measurements to parameter_id and sample_id
fact_measurement = df_long.merge(dim_parameter[["parameter_id", "parameter_std"]], on="parameter_std", how="left")

# quick sanity
print("dim_parameter:", dim_parameter.shape)
print("dim_location:", dim_location.shape)
print("fact_sample:", fact_sample.shape)
print("fact_measurement:", fact_measurement.shape)

display(dim_parameter.head(5))
display(dim_location.head(3))
display(fact_sample.head(3))
display(fact_measurement.head(3))

dim_parameter: (13, 3)
dim_location: (243, 6)
fact_sample: (6854, 8)
fact_measurement: (67630, 23)


,parameter_id,parameter_std,canonical_unit
0,1,Ammoniacal Nitrogen,mg N-NH3-/L
1,2,BOD5,mg O2/L
2,3,COD,mg O2/L
3,4,Chloride,mg Cl-/L
4,5,Dissolved Oxygen,mg O2/L


,location_id,NOMBRE DEL PUNTO DE MONITOREO,LATITUD,LONGITUD,DEPARTAMENTO,MUNICIPIO
0,1,RCA_BOGOTA_CUN_VILLAPINZON_PTE.CARRETERA-BOGOTA,5.218611,-73.595556,CUNDINAMARCA,VILLAPINZÓN
1,2,RCA_BOGOTA_CUN_TOCANCIPA_PTE.TULIO BOTERO-BOGOTA,4.971917,-73.916139,CUNDINAMARCA,TOCANCIPÁ
2,3,RCA_BOGOTA_CUN_VILLAPINZON_SANPEDRO-BOGOTA,5.194722,-73.613889,CUNDINAMARCA,VILLAPINZÓN


,CODIGO__MUESTRA,FECHA,NOMBRE DEL PUNTO DE MONITOREO,LATITUD,LONGITUD,DEPARTAMENTO,MUNICIPIO,location_id
0,11284,2005-02-15,RCA_BOGOTA_CUN_VILLAPINZON_PTE.CARRETERA-BOGOTA,5.218611,-73.595556,CUNDINAMARCA,VILLAPINZÓN,1
1,11285,2005-02-15,RCA_BOGOTA_CUN_TOCANCIPA_PTE.TULIO BOTERO-BOGOTA,4.971917,-73.916139,CUNDINAMARCA,TOCANCIPÁ,2
2,11289,2005-02-15,RCA_BOGOTA_CUN_VILLAPINZON_SANPEDRO-BOGOTA,5.194722,-73.613889,CUNDINAMARCA,VILLAPINZÓN,3


,Unnamed: 0,NOMBRE DEL PUNTO DE MONITOREO,LATITUD,LONGITUD,ELEVACIÓN (m.s.n.m.),CORRIENTE,ZONA HIDROGRÁFICA - ZH,SZH - Código (#Área#Zona##Subzona),Nombre Subzona Hidrográfica,DEPARTAMENTO,...,*RESULTADO,UNIDAD DEL RESULTADO,PROYECTO,CODIGO__MUESTRA,parameter_std,value_raw,is_censored,value_num,unit_raw,parameter_id
0,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,...,60,µS/cm,Ideam,14615,Electrical Conductivity,60,False,60.0,µS/cm,6
1,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,...,85.7,µS/cm,Ideam,20148,Electrical Conductivity,85.7,False,85.7,µS/cm,6
2,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,...,109.2,µS/cm,Ideam,22820,Electrical Conductivity,109.2,False,109.2,µS/cm,6


## 3. SQLite Database Build

This section creates the SQLite database, writes dimension and fact tables, and adds indexes
to support fast analytical queries.

In [ ]:
# Build SQLite database
if DB_PATH.exists():
    DB_PATH.unlink()  # recreate cleanly for reproducibility

conn = sqlite3.connect(DB_PATH)

# Write tables
dim_parameter.to_sql("dim_parameter", conn, index=False)
dim_location.to_sql("dim_location", conn, index=False)
fact_sample.to_sql("fact_sample", conn, index=False)
fact_measurement.to_sql("fact_measurement", conn, index=False)

# Indexes (speed!)
cur = conn.cursor()

cur.executescript("""
CREATE INDEX idx_fact_measurement_sample ON fact_measurement(CODIGO__MUESTRA);
CREATE INDEX idx_fact_measurement_param  ON fact_measurement(parameter_id);
CREATE INDEX idx_fact_sample_location    ON fact_sample(location_id);
CREATE INDEX idx_fact_sample_date        ON fact_sample(FECHA);
""")

conn.commit()

# Sanity checks
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
display(tables)

row_counts = {}
for t in ["dim_parameter", "dim_location", "fact_sample", "fact_measurement"]:
    row_counts[t] = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t};", conn)["n"].iloc[0]

print("Row counts:", row_counts)

conn.close()
print(f"✅ SQLite DB created: {DB_PATH}")


,name
0,dim_location
1,dim_parameter
2,fact_measurement
3,fact_sample


Row counts: {'dim_parameter': np.int64(13), 'dim_location': np.int64(243), 'fact_sample': np.int64(6854), 'fact_measurement': np.int64(67630)}
✅ SQLite DB created: D:\Documents\Portfolio\01-water-quality-normative\data\water_quality_analysis.db


## 4. Quick Validation Queries

Basic SQL queries are executed to confirm integrity and expected coverage.

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Example: top parameters by record count
q1 = """
SELECT p.parameter_std, COUNT(*) AS n
FROM fact_measurement m
JOIN dim_parameter p ON m.parameter_id = p.parameter_id
GROUP BY p.parameter_std
ORDER BY n DESC
LIMIT 15;
"""
display(pd.read_sql_query(q1, conn))

# Example: sample count
q2 = "SELECT COUNT(DISTINCT CODIGO__MUESTRA) AS unique_samples FROM fact_sample;"
display(pd.read_sql_query(q2, conn))

conn.close()

,parameter_std,n
0,Electrical Conductivity,6771
1,pH,6764
2,Temperature,6738
3,COD,6694
4,Total Suspended Solids,6660
5,Dissolved Oxygen,6559
6,Turbidity,6549
7,Nitrite,5129
8,Nitrate,5101
9,Ammoniacal Nitrogen,4997


,unique_samples
0,6818


## Phase 2 Checkpoint

A relational SQLite database has been built from Phase 1 outputs, including parameter and location
dimensions, sample metadata, and long-format measurements with full traceability.

This database enables efficient querying and provides the foundation for Phase 3 (normative compliance rules)
and Phase 4 (risk analysis).